In [1]:
using Plots
using Revise
using DataInterpolations
using RegularizationTools
using NaturalNeighbours

In [2]:
function read_custom_file(filepath::String)
    array1 = Float64[]
    array2 = Float64[]

    start_reading = false

    num_regex = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    open(filepath, "r") do file
        for line in eachline(file)

            clean_line = strip(line)

            if startswith(clean_line, "+") || startswith(clean_line, "-")
                matches = [m.match for m in eachmatch(num_regex, clean_line)]

                if length(matches) >= 2
                    push!(array1, parse(Float64, matches[1]))
                    push!(array2, parse(Float64, matches[2]))
                end
            else
                continue
            end
        end
    end
    return array1, array2
end

read_custom_file (generic function with 1 method)

In [3]:
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/2.S9272.PtCoCu.7/S9272-FORC-50-2000-5s"
cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/3.S9279.PtCoCu.10/S9279-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/6.S9281.CoPt.10/S9281-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/4.S9283.CoPt.6/S9283-FORC-50-1500-5s"
H_read = Float64[]
M_read = Float64[]
H_read, M_read = read_custom_file(cale * ".txt")

([1925.586, 1965.563, 2005.763, 1845.309, 1899.857, 1953.748, 2005.701, 1764.73, 1819.346, 1873.425  …  1516.839, 1570.765, 1625.583, 1679.412, 1733.289, 1788.078, 1841.943, 1895.93, 1950.749, 2004.601], [0.0001844221, 0.0001863384, 0.0001876805, 0.0001817743, 0.000184455, 0.0001862461, 0.0001876175, 0.0001791379, 0.0001813849, 0.0001837609  …  0.0001580386, 0.000160169, 0.000167161, 0.0001730312, 0.0001726415, 0.0001768686, 0.0001817618, 0.0001832247, 0.000185166, 0.0001896564])

In [4]:
#Normalize
normH = 1.0e3
normM = 1.0e-3
H_read = H_read ./ normH
M_read = M_read ./ normM
H_read, M_read

([1.925586, 1.9655630000000002, 2.005763, 1.8453089999999999, 1.899857, 1.953748, 2.005701, 1.76473, 1.819346, 1.873425  …  1.516839, 1.5707650000000002, 1.625583, 1.6794120000000001, 1.733289, 1.788078, 1.8419429999999999, 1.8959300000000001, 1.950749, 2.004601], [0.1844221, 0.1863384, 0.1876805, 0.1817743, 0.184455, 0.1862461, 0.18761750000000002, 0.17913790000000002, 0.1813849, 0.1837609  …  0.1580386, 0.16016899999999998, 0.167161, 0.1730312, 0.1726415, 0.1768686, 0.1817618, 0.1832247, 0.185166, 0.18965639999999998])

In [5]:
scatter(H_read, M_read, 
    title = "Scatter Plot of Data", 
    xlabel = "H", 
    ylabel = "M", 
    legend = false,   
    markersize = 3, 
    markercolor = :indianred,
    markerstrokecolor =:indianred
)

In [6]:
# Detect FORCs (Hr, numPointsPerFORC)
Hr_read = Float64[]
numPointsPerFORC = Int[]

current_Hr = H_read[1]
push!(Hr_read, current_Hr)

counterPointsPerFORC = 1

for i in 2:(length(H_read))
    if H_read[i] < H_read[i-1]
        global current_Hr = H_read[i]
        push!(numPointsPerFORC, counterPointsPerFORC)
        global counterPointsPerFORC = 0
    end
    global counterPointsPerFORC += 1
    push!(Hr_read, current_Hr)
end
push!(numPointsPerFORC, counterPointsPerFORC)
numFORCs = length(numPointsPerFORC)

println("----------- Total $(length(numPointsPerFORC)) FORCs -----------")
println("---------  $(numPointsPerFORC[1]) first / $(numPointsPerFORC[numFORCs]) last ---------")

----------- Total 50 FORCs -----------
---------  3 first / 75 last ---------


In [7]:
# Save Hr-H-M original file
n = length(Hr_read)
if length(H_read) != n || length(M_read) != n
    error("Error: All three arrays must have the same length.")
end

open(cale * "_Hr-H-M_orig.dat", "w") do file
    for i in 1:n
        println(file, "$(Hr_read[i])  $(H_read[i])  $(M_read[i])")
    end
end

In [8]:
function gimmeOneFORC(theFORC::Int64)
    contorNumPoints = 0
    H_interp = Float64[]
    M_interp = Float64[]
    #detect indexes for $(theFORC)
    if theFORC > 1
        for i in 1:theFORC-1
            contorNumPoints += numPointsPerFORC[i]
        end
    else
        contorNumPoints = 0
    end

    #myHr = Hr_read[contorNumPoints]
    startPointOnFORC = contorNumPoints + 1
    push!(H_interp, H_read[startPointOnFORC])
    push!(M_interp, M_read[startPointOnFORC])
    contorNumPoints = 1
    while (Hr_read[startPointOnFORC+contorNumPoints-1] - Hr_read[startPointOnFORC+contorNumPoints]) < 1.0e-5 #eps - compare floats
        push!(H_interp, H_read[startPointOnFORC+contorNumPoints])
        push!(M_interp, M_read[startPointOnFORC+contorNumPoints])
        contorNumPoints += 1
        if (startPointOnFORC + contorNumPoints) > length(Hr_read)
            break
        end
    end
    H_interp, M_interp
end

gimmeOneFORC (generic function with 1 method)

In [9]:
showTest = true

true

In [10]:
if (showTest)
    # Interpolate one FORC
    plotInterpFORC = numFORCs
    println("----------------- Interpolating Akima $(plotInterpFORC)-th FORC ----------------")
    H_interp, M_interp = gimmeOneFORC(plotInterpFORC)
    Example = AkimaInterpolation(M_interp, H_interp)
    plot(Example)
end

----------------- Interpolating Akima 50-th FORC ----------------


In [11]:
if (showTest)
    # Smooth one FORC (N - num points after smoothing)
    t, u = gimmeOneFORC(plotInterpFORC)
    d = 3 + div(plotInterpFORC, 3) - 1
    A = RegularizationSmooth(u, t, d; alg=:gcv_svd)
    û = A.û
    N = plotInterpFORC #length(t)
    titp = collect(range(minimum(t), maximum(t), length=N))
    uitp = A.(titp)
    scatter(t, u, label="simulated data", legend=:top)
    plot!(titp, uitp, label="smoothed interpolation")
end

In [12]:
if (showTest)
    # Save interpolated / smoothed file for a single FORC
    nSmooth = length(titp)
    nOrig = length(H_interp)
    n = maximum([nSmooth, nOrig])

    open(cale * "_InterpSmoothCheck.dat", "w") do file
        for i in 1:n
            if (nSmooth >= nOrig)
                if (i <= nOrig)
                    println(file, "$(t[i]), $(u[i]), $(H_interp[i]), $(M_interp[i]), $(titp[i]), $(uitp[i])")
                else
                    println(file, "       ,        ,              ,                , $(titp[i]), $(uitp[i])")
                end
            else
                if (i <= nSmooth)
                    println(file, "$(t[i]), $(u[i]), $(H_interp[i]), $(M_interp[i]), $(titp[i]), $(uitp[i])")
                else
                    println(file, "$(t[i]), $(u[i]), $(H_interp[i]), $(M_interp[i]),           ,           ")
                end
            end
            
        end
    end
end

In [13]:
numFORCs = length(numPointsPerFORC) + 1 # +1 pentru one-point-FORC

M_NoExtend  = zeros(Float64, numFORCs, numFORCs) #matricea M
M_ExtendLine = zeros(Float64, numFORCs, numFORCs) #matricea M
M_ExtendMirror = zeros(Float64, numFORCs, numFORCs) #matricea M
Hr_NoExtend = zeros(Float64, numFORCs)
H_NoExtend  = zeros(Float64, numFORCs)

H, M = gimmeOneFORC(1)
Hr_NoExtend[1] = H[end]
H_NoExtend[numFORCs] = H[end]
M_NoExtend[1, numFORCs] = M[end]
M_ExtendLine[1, numFORCs] = M[end]
M_ExtendMirror[1, numFORCs] = M[end]
for i in 1:numFORCs-1
    M_ExtendLine[1, i] = M[end]
    M_ExtendMirror[1, i] = M[end]
end

for i in 2:numFORCs
    H, M = gimmeOneFORC(i-1)
    Hr_NoExtend[i] = H[begin]
    H_NoExtend[numFORCs-i+1] = H[begin]
end

In [14]:
for i in 2:numFORCs
    t, u = gimmeOneFORC(i-1)
    if t[end] < Hr_NoExtend[1]
        t[end] = Hr_NoExtend[1]
    end

    #Example = AkimaInterpolation(M_interp, H_interp)

    d = 3 + div(i, 3)-1
    A = RegularizationSmooth(u, t, d; alg=:gcv_svd)
    û = A.û
    N = i #length(t)
    titp = [Hr_NoExtend[j] for j=i:-1:1] #collect(range(minimum(t), maximum(t), length=N))
    uitp = A.(titp)
    #println("---------- $(i)/$(numFORCs) -------------")

    for j in 1:i
        M_NoExtend[i, numFORCs-i+j] = uitp[j]
        M_ExtendLine[i, numFORCs-i+j] = uitp[j]
        M_ExtendMirror[i, numFORCs-i+j] = uitp[j]
        #println(" $(M_NoExtend[i, numFORCs-i+j]) ")
    end
    for j in 1:numFORCs-i
        M_ExtendLine[i, j] = uitp[begin]
    end
    for j in numFORCs-i+1:-1:1
        if j+(2*(numFORCs-i+1-j)+1) <= numFORCs
            M_ExtendMirror[i, j] = M_ExtendMirror[i, j+(2*(numFORCs-i+1-j)+1)]
        else
            M_ExtendMirror[i, j] = uitp[end]
        end
    end

end

In [15]:
if (showTest)
    heatmap(M_ExtendMirror, yflip=true)
end

In [16]:
if (showTest)
        scatter(H_NoExtend, transpose(M_ExtendLine), legend=false,
                                                        markersize=3,
                                                        markercolor=:indianred,
                                                        markerstrokecolor=:indianred)
end

In [17]:
open(cale * ".MGRID.REGULAR.dat", "w") do file
    for i in 1:numFORCs
        for j in numFORCs-i+1:numFORCs
            println(file, "$(Hr_NoExtend[i])  $(H_NoExtend[j])  $(M_NoExtend[i, j])")
        end
        print(file, "\n")
    end
end

In [18]:
open(cale * ".MGRID.PrelungLine.dat", "w") do file
    for i in 1:numFORCs
        for j in 1:numFORCs
            println(file, "$(Hr_NoExtend[i])  $(H_NoExtend[j])  $(M_ExtendLine[i, j])")
        end
        print(file, "\n")
    end
end

In [19]:
SP = 4
FORC_NoExtend = zeros(Float64, numFORCs, numFORCs) #matricea FORC 
;

In [20]:
itp = interpolate(Hr_read, H_read, M_read; derivatives=true)

Natural Neighbour Interpolant
    z: [0.1844221, 0.1863384, 0.1876805, 0.1817743, 0.184455, 0.1862461, 0.18761750000000002, 0.17913790000000002, 0.1813849, 0.1837609  …  0.1580386, 0.16016899999999998, 0.167161, 0.1730312, 0.1726415, 0.1768686, 0.1817618, 0.1832247, 0.185166, 0.18965639999999998]
    ∇: [(-0.022763333788351697, 0.05273368837986039), (-0.0043030116177312635, 0.03944633480863641), (-0.0013083339118928972, 0.030797551329989965), (-0.017805603723898136, 0.051137427280990556), (-0.0066946489389105865, 0.03961491522961876), (-0.00023142775870704639, 0.030099266716217942), (-0.0020087832030059294, 0.03751283245328574), (-0.0026918808160084668, 0.04519657170340061), (-0.0029701751490367172, 0.043760432343262656), (-0.0012472674558427223, 0.034437719615084204)  …  (-0.016057979358079694, 0.08054510434265391), (0.02897368245664777, 0.08478404313077662), (-0.0004902821776500368, 0.10373520377341859), (-0.02437380166488359, 0.060995614294190925), (0.037901460531544966, 0.040350620

In [21]:
H_ForInterp = Float64[]
Hr_ForInterp = Float64[]

for i in 1:numFORCs
    for j in numFORCs-i+1:numFORCs
        push!(Hr_ForInterp, Hr_NoExtend[i])
        push!(H_ForInterp, H_NoExtend[j])
    end
end

In [22]:
sibson_1_vals = itp(Hr_ForInterp, H_ForInterp; method=Sibson(1));
;

In [23]:
open(cale * ".INTERP.dat", "w") do file
    for i in 1:length(Hr_ForInterp)
        if (i>2) && (Hr_ForInterp[i] < Hr_ForInterp[i-1])
            println(file, "")
        end
        println(file, "$(Hr_ForInterp[i])  $(H_ForInterp[i])  $(sibson_1_vals[i])")
        

    end
end